# Model Training

Notebook version that reuses the training utilities from `clearsky_lstm.py`.


In [ ]:
from types import SimpleNamespace
import os
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split

from data import NEXRADDataset
from clearsky_lstm import (
    ConvLSTMForecaster,
    SmaAtUNet,
    device,
    train_one_epoch,
    evaluate,
)


In [ ]:
RAW_DATA_PATH = "data/raw"
CACHE_DATA_PATH = "data/cache"
STATION = "KMAX"
VAL_FRAC = 0.1
TEST_FRAC = 0.1
SEED = 13
BATCH_SIZE = 8
NUM_WORKERS = 4
MODEL = "smaat_unet"
T_IN = 6
T_OUT = 6
HIDDEN_CHANNELS = [64, 64, 64]
NUM_LAYERS = 3
TEACHER_FORCING = 0.0
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
EPOCHS = 20
SAVE_SAMPLES = False
SAMPLE_DIR = "samples"
MODEL_OUT = "checkpoints/final_model.pt"

args = SimpleNamespace(
    model=MODEL,
    stations=[STATION],
    t_in=T_IN,
    t_out=T_OUT,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    hidden_ch=HIDDEN_CHANNELS,
    num_layers=NUM_LAYERS,
    teacher_forcing=TEACHER_FORCING,
    save_samples=SAVE_SAMPLES,
    sample_dir=SAMPLE_DIR,
    model_out=MODEL_OUT,
    seed=SEED,
)

print(f"Using device: {device}")
print(args)


In [ ]:
print("Building dataset...")
ds = NEXRADDataset(
    raw_root=RAW_DATA_PATH,
    stations=args.stations,
    t_in=args.t_in,
    t_out=args.t_out,
    cache_root=CACHE_DATA_PATH,
)

n = len(ds)
n_val = int(args.val_frac * n)
n_test = int(args.test_frac * n)
n_train = n - n_val - n_test
print(f"Dataset size: {n}")
print(f"Train/val/test split: {n_train}/{n_val}/{n_test}")

torch.manual_seed(args.seed)
train_ds, val_ds, test_ds = random_split(ds, [n_train, n_val, n_test])

train_loader = DataLoader(
    train_ds,
    batch_size=args.batch_size,
    shuffle=True,
    num_workers=args.num_workers,
)
val_loader = DataLoader(
    val_ds,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
)
test_loader = DataLoader(
    test_ds,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
)

print("Dataloaders ready.")


In [ ]:
if args.model == "smaat_unet":
    model = SmaAtUNet(in_channels=args.t_in, out_channels=args.t_out)
else:
    model = ConvLSTMForecaster(hidden_ch=args.hidden_ch, num_layers=args.num_layers)

model = model.to(device)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=args.lr,
    weight_decay=args.weight_decay,
)
criterion = torch.nn.L1Loss()

print(model.__class__.__name__)
print("Model built and optimizer initialized.")


In [ ]:
history = {"train_loss": [], "val_loss": []}

print("Beginning training...")
for epoch in range(args.epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, args)
    val_loss = evaluate(model, val_loader, criterion, device, args, epoch)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"Epoch {epoch + 1}/{args.epochs} | train: {train_loss:.3f} | val: {val_loss:.3f}")


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("Epoch")
plt.ylabel("L1 Loss")
plt.title("Training History")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
print("Evaluating model...")
test_loss = evaluate(model, test_loader, criterion, device, args)
print(f"Final loss on test set: {test_loss:.3f}")

model_dir = os.path.dirname(args.model_out)
if model_dir:
    os.makedirs(model_dir, exist_ok=True)
torch.save(model.state_dict(), args.model_out)
print(f"Saved final model parameters to {args.model_out}")
